About Dataset
Context This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content 5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset. Columns

asin - ID of the product, like B000FA64PK
helpful - helpfulness rating of the review - example: 2/3.
overall - rating of the product.
reviewText - text of the review (heading).
reviewTime - time of the review (raw).
reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
reviewerName - name of the reviewer.
summary - summary of the review (description).
unixReviewTime - unix timestamp.
Acknowledgements This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

Inspiration

- Sentiment analysis on reviews.
- Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.
- Fake reviews/ outliers.
- Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
- Any other interesting analysis


Best Practises
- Preprocessing And Cleaning
- Train Test Split
- BOW,TFIDF,Word2vec
- Train ML algorithms

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

In [2]:
from bs4 import BeautifulSoup
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jenis\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jenis\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB         
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score,
                             classification_report, roc_auc_score)

In [4]:
data = pd.read_csv('Kindle Reviews/all_kindle_review.csv')

In [5]:
df = data[['reviewText', 'rating']]

In [6]:
print("Shape before cleaning:", df.shape)

Shape before cleaning: (12000, 2)


In [7]:
df.dropna(subset=['reviewText', 'rating'], inplace=True)
print("Shape after dropping nulls:", df.shape)
print("Rating distribution:\n", df['rating'].value_counts())

Shape after dropping nulls: (12000, 2)
Rating distribution:
 rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64


In [8]:
df['sentiment'] = df['rating'].apply(lambda x: 0 if x < 3 else 1)
print("\nSentiment distribution:\n", df['sentiment'].value_counts())


Sentiment distribution:
 sentiment
1    8000
0    4000
Name: count, dtype: int64


In [9]:
STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [10]:
def clean_text(text: str) -> str:
    """Full cleaning pipeline: lowercase → HTML → URLs → special chars → stopwords → lemmatize."""
    if not isinstance(text, str):          # guard against NaN / non-string
        return ""
    text = text.lower()
    text = BeautifulSoup(text, 'lxml').get_text()                        # strip HTML
    text = re.sub(r'https?://\S+|ftp://\S+', '', text)                   # remove URLs
    text = re.sub(r'[^a-z0-9 ]', ' ', text)                              # keep alphanumeric
    tokens = [lemmatizer.lemmatize(w) for w in text.split()
              if w not in STOP_WORDS and len(w) > 2]                     # remove short tokens too
    return " ".join(tokens)

In [11]:
df['clean_review'] = df['reviewText'].apply(clean_text)

In [12]:
df = df[df['clean_review'].str.strip() != ""]
print("\nShape after text cleaning:", df.shape)
print(df[['reviewText', 'clean_review', 'sentiment']].head(3))



Shape after text cleaning: (12000, 4)
                                          reviewText  \
0  Jace Rankin may be short, but he's nothing to ...   
1  Great short read.  I didn't want to put it dow...   
2  I'll start by saying this is the first of four...   

                                        clean_review  sentiment  
0  jace rankin may short nothing mess man hauled ...          1  
1  great short read want put read one sitting sex...          1  
2  start saying first four book expecting conclud...          1  


In [13]:
df.head()

,reviewText,rating,sentiment,clean_review
0,"Jace Rankin may be short, but he's nothing to ...",3,1,jace rankin may short nothing mess man hauled ...
1,Great short read. I didn't want to put it dow...,5,1,great short read want put read one sitting sex...
2,I'll start by saying this is the first of four...,3,1,start saying first four book expecting conclud...
3,Aggie is Angela Lansbury who carries pocketboo...,3,1,aggie angela lansbury carry pocketbook instead...
4,I did not expect this type of book to be in li...,4,1,expect type book library pleased find price right


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'], df['sentiment'],
    test_size=0.2, random_state=42, stratify=df['sentiment']   # ← was missing before
)

In [15]:
bow   = CountVectorizer(max_features=30000, ngram_range=(1, 2))   # bigrams added
tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), sublinear_tf=True)

In [16]:
X_train_bow   = bow.fit_transform(X_train)
X_test_bow    = bow.transform(X_test)


In [17]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

In [18]:
models = {
    "Multinomial NB":       MultinomialNB(),
    "Logistic Regression":  LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs'),
    "Random Forest":        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

In [19]:
def evaluate(name, model, X_tr, X_te, y_tr, y_te, vec_name):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None

    print(f"\n{'='*55}")
    print(f" {name}  [{vec_name}]")
    print(f"{'='*55}")
    print(f"Accuracy : {accuracy_score(y_te, y_pred):.4f}")
    if y_prob is not None:
        print(f"ROC-AUC  : {roc_auc_score(y_te, y_prob):.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_te, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_te, y_pred, target_names=['Negative', 'Positive']))

In [20]:
for model_name, model in models.items():
    evaluate(model_name, model, X_train_bow,   X_test_bow,   y_train, y_test, "BOW")
    evaluate(model_name, model, X_train_tfidf, X_test_tfidf, y_train, y_test, "TF-IDF")



 Multinomial NB  [BOW]
Accuracy : 0.8600
ROC-AUC  : 0.9160

Confusion Matrix:
[[ 656  144]
 [ 192 1408]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.77      0.82      0.80       800
    Positive       0.91      0.88      0.89      1600

    accuracy                           0.86      2400
   macro avg       0.84      0.85      0.84      2400
weighted avg       0.86      0.86      0.86      2400


 Multinomial NB  [TF-IDF]
Accuracy : 0.7558
ROC-AUC  : 0.9259

Confusion Matrix:
[[ 237  563]
 [  23 1577]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.91      0.30      0.45       800
    Positive       0.74      0.99      0.84      1600

    accuracy                           0.76      2400
   macro avg       0.82      0.64      0.65      2400
weighted avg       0.80      0.76      0.71      2400


 Logistic Regression  [BOW]
Accuracy : 0.8554
ROC-AUC  : 0.9123

Confusion Matrix:

In [21]:
import numpy as np
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

In [22]:
X_train_tokens = [text.split() for text in X_train]
X_test_tokens  = [text.split() for text in X_test]

In [23]:
w2v_model = Word2Vec(
    sentences   = X_train_tokens,
    vector_size = 100,
    window      = 5,
    min_count   = 2,
    workers     = 4,
    sg          = 0,     
    epochs      = 10
)

In [24]:
print(f"Vocabulary size: {len(w2v_model.wv)}")

Vocabulary size: 14177


In [25]:
print("\nWords similar to 'good':")
print(w2v_model.wv.most_similar('good', topn=5))


Words similar to 'good':
[('decent', 0.8102561831474304), ('great', 0.7466874718666077), ('hilariously', 0.6776085495948792), ('dissappointed', 0.6633038520812988), ('alot', 0.6623838543891907)]


In [26]:
def get_avg_vector(tokens: list, model: Word2Vec, vector_size: int) -> np.ndarray:
    """Return the mean Word2Vec vector for a list of tokens.
       Returns a zero vector if no tokens are in the vocabulary."""
    vectors = [
        model.wv[word] for word in tokens if word in model.wv
    ]
    if vectors:
        return np.mean(vectors, axis=0)
    return np.zeros(vector_size)

In [27]:
VECTOR_SIZE = w2v_model.vector_size

X_train_w2v = np.array([get_avg_vector(tokens, w2v_model, VECTOR_SIZE) for tokens in X_train_tokens])
X_test_w2v  = np.array([get_avg_vector(tokens, w2v_model, VECTOR_SIZE) for tokens in X_test_tokens])

In [ ]:
w2v_models = {
    "Logistic Regression" : LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs'),
    "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gaussian NB"         : GaussianNB(),   
}

In [29]:
for model_name, model in w2v_models.items():
    evaluate(model_name, model, X_train_w2v, X_test_w2v, y_train, y_test, "Word2Vec")


 Logistic Regression  [Word2Vec]
Accuracy : 0.8083
ROC-AUC  : 0.8832

Confusion Matrix:
[[ 497  303]
 [ 157 1443]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.76      0.62      0.68       800
    Positive       0.83      0.90      0.86      1600

    accuracy                           0.81      2400
   macro avg       0.79      0.76      0.77      2400
weighted avg       0.80      0.81      0.80      2400


 Random Forest  [Word2Vec]
Accuracy : 0.7963
ROC-AUC  : 0.8592

Confusion Matrix:
[[ 479  321]
 [ 168 1432]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.74      0.60      0.66       800
    Positive       0.82      0.90      0.85      1600

    accuracy                           0.80      2400
   macro avg       0.78      0.75      0.76      2400
weighted avg       0.79      0.80      0.79      2400


 Gaussian NB  [Word2Vec]
Accuracy : 0.7342
ROC-AUC  : 0.8218

Confusion